# Chapter 3 Lab: The Frame Operator

This notebook accompanies **Chapter 3**. It builds the frame-operator
pipeline (`frame_operator`, `exact_frame_bounds`, `is_tight`) using
exact eigenvalue computation, and works through all six lab exercises.

Run the setup cell first.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.set_printoptions(precision=4, suppress=True)
%matplotlib inline

def illumination(x, vectors):
    '''Compute sum_i |<x, f_i>|^2 for a list/array of vectors. (From Chapter 1.)'''
    return sum(np.dot(x, f)**2 for f in vectors)

def frame_operator(F):
    '''Frame operator S = F @ F.T (real frames). F has shape (n, m).'''
    return F @ F.T

def exact_frame_bounds(F):
    '''Exact optimal frame bounds via eigenvalues of S = F F^T.'''
    S = frame_operator(F)
    eigenvalues, eigenvectors = np.linalg.eigh(S)
    return float(eigenvalues[0]), float(eigenvalues[-1]), eigenvalues, eigenvectors

def is_tight(F, tol=1e-10):
    A, B, _, _ = exact_frame_bounds(F)
    return (B - A) < tol * max(1.0, B)

def sampling_bounds(F, num_samples=100_000, seed=0):
    n = F.shape[0]
    rng = np.random.default_rng(seed)
    vals = []
    for _ in range(num_samples):
        x = rng.standard_normal(n)
        x = x / np.linalg.norm(x)
        vals.append(x @ frame_operator(F) @ x)
    vals = np.array(vals)
    return vals.min(), vals.max()

## Lab Exercise 3.1: The Full Frame Operator Pipeline

Run `frame\_operator`, `exact\_frame\_bounds`, and `is\_tight` on all six frames from this chapter: (a) triangular, (b) $\{h_1,h_2,h_3\}$, (c) standard ONB, (d) non-spanning $\{g_1,g_2\}$ in $\mathbb{R}^3$, (e) square vertices, (f) $\{(1,0),(0,2)\}$. Verify the trace formula for each.

In [ ]:
f1 = np.array([1.0, 0.0])
f2 = np.array([-0.5, np.sqrt(3)/2])
f3 = np.array([-0.5, -np.sqrt(3)/2])
F_tri = np.column_stack([f1, f2, f3])

h1, h2, h3 = np.array([1.0,0.0]), np.array([0.0,1.0]), np.array([1.0,1.0])
F_h = np.column_stack([h1, h2, h3])

F_onb = np.column_stack([np.array([1.0,0.0]), np.array([0.0,1.0])])

g1, g2 = np.array([1.0,0.0,0.0]), np.array([0.0,1.0,0.0])
F_g = np.column_stack([g1, g2])

angles4 = np.array([0, np.pi/2, np.pi, 3*np.pi/2])
F_sq = np.vstack([np.cos(angles4), np.sin(angles4)])

F_e2 = np.column_stack([np.array([1.0,0.0]), np.array([0.0,2.0])])

frames = [
    ("Triangular", F_tri), ("{h1,h2,h3}", F_h), ("Standard ONB", F_onb),
    ("Non-spanning {g1,g2}", F_g), ("Square", F_sq), ("{e1, 2e2}", F_e2),
]

for name, F in frames:
    S = None       # TODO: frame_operator(F)
    A, B, eigs, _ = None, None, None, None  # TODO: exact_frame_bounds(F)
    tight = None   # TODO: is_tight(F)

    print(f"--- {name} ---")
    print(f"S =\n{S}")
    print(f"eigenvalues = {eigs}, A={A:.4f}, B={B:.4f}, tight={tight}")

    trace_S = None          # TODO: np.trace(S)
    trace_direct = None     # TODO: sum(np.dot(F[:,i], F[:,i]) for i in range(F.shape[1]))
    print(f"trace(S)={trace_S:.4f}, sum||fi||^2={trace_direct:.4f}, match={np.isclose(trace_S, trace_direct)}\n")

## Lab Exercise 3.2: Tightness from the Trace

Verify $A=m/n$ for the triangular frame ($m=3,n=2$) and square ($m=4,n=2$). Then answer: for $m=6$ unit-norm vectors in $\mathbb{R}^3$, what must $A$ equal?

In [ ]:
A_tri, _, _, _ = exact_frame_bounds(F_tri)
print(f"Triangle: A={A_tri:.4f}, m/n={3/2:.4f}, match={np.isclose(A_tri, 3/2)}")

A_sq, _, _, _ = exact_frame_bounds(F_sq)
print(f"Square:   A={A_sq:.4f}, m/n={4/2:.4f}, match={np.isclose(A_sq, 4/2)}")

A_required_m6_n3 = None  # TODO: fill in m/n for m=6, n=3
print(f"\nFor m=6, n=3 tight unit-norm frame: A must equal {A_required_m6_n3}")

## Lab Exercise 3.3: Reconstruction Accuracy

(a) For five random $x\in\mathbb{R}^2$, reconstruct via the triangular frame's canonical dual and report the error. (b) Corrupt $c_1$ by $\delta=0.1,0.5,1.0$ and report the error. (c) Repeat (b) for the basis $\{e_1,e_2\}$ and compare robustness.

In [ ]:
rng = np.random.default_rng(42)
X = rng.standard_normal((2, 5))
A_tri = 1.5

print("=== Part (a): exact reconstruction ===")
for i in range(5):
    x = X[:, i]
    c = None        # TODO: array of <x, f1>, <x, f2>, <x, f3>
    x_hat = None     # TODO: (1/A_tri) * sum_j c_j * f_j
    print(f"x={x.round(4)}: ||x_hat - x|| = {np.linalg.norm(x_hat - x):.2e}")

print("\n=== Part (b): corrupted coefficient, triangular frame ===")
x = X[:, 0]
c_true = np.array([np.dot(x, f1), np.dot(x, f2), np.dot(x, f3)])
for delta in [0.1, 0.5, 1.0]:
    c_corrupt = c_true.copy()
    c_corrupt[0] += delta
    x_hat_delta = None      # TODO: reconstruct using c_corrupt
    err = np.linalg.norm(x_hat_delta - x)
    print(f"delta={delta}: ||x_hat_delta - x|| = {err:.4f}  (err/delta = {err/delta:.4f})")

print("\n=== Part (c): corrupted coefficient, standard basis {e1,e2} ===")
e1, e2 = np.array([1.0,0.0]), np.array([0.0,1.0])
for delta in [0.1, 0.5, 1.0]:
    c1_true = np.dot(x, e1)
    c2_true = np.dot(x, e2)
    c1_corrupt = c1_true + delta
    x_hat_basis = None  # TODO
    err_basis = np.linalg.norm(x_hat_basis - x)
    print(f"delta={delta}: ||x_hat_delta - x|| = {err_basis:.4f}  (err/delta = {err_basis/delta:.4f})")

*Your comparison: which frame is more robust to the same perturbation, and why?* (replace this text)

## Lab Exercise 3.4: Exact vs. Sampled Bounds

Run `exact\_frame\_bounds` and `sampling\_bounds` on the triangular frame and $\{h_1,h_2,h_3\}$. Why does the sampled estimate always satisfy $A_{\text{sampled}}\ge A_{\text{exact}}$, $B_{\text{sampled}}\le B_{\text{exact}}$?

In [ ]:
for name, F in [("Triangular", F_tri), ("{h1,h2,h3}", F_h)]:
    A_ex, B_ex, _, _ = None, None, None, None  # TODO: exact_frame_bounds(F)
    A_samp, B_samp = None, None                # TODO: sampling_bounds(F, num_samples=100_000)
    print(f"{name}: A_exact={A_ex:.6f}, A_sampled={A_samp:.6f}  |  "
          f"B_exact={B_ex:.6f}, B_sampled={B_samp:.6f}")

*Your explanation: why is A_sampled >= A_exact and B_sampled <= B_exact always?* (replace this text)

## Lab Exercise 3.5: Eigenvectors as Extremal Directions

For $\{h_1,h_2,h_3\}$ ($A=1,B=3$): retrieve $u_1,u_2$; verify $\langle Su_1, u_1 \rangle=3$, $\langle Su_2, u_2 \rangle=1$; plot the illumination function marking these directions.

In [ ]:
A_h, B_h, eigs_h, vecs_h = exact_frame_bounds(F_h)
print(f"Eigenvalues: {eigs_h}")

u1 = None  # TODO: eigenvector for LARGER eigenvalue (vecs_h[:, -1])
u2 = None  # TODO: eigenvector for SMALLER eigenvalue (vecs_h[:, 0])

S_h = frame_operator(F_h)
rayleigh_u1 = None  # TODO: u1 @ S_h @ u1
rayleigh_u2 = None  # TODO: u2 @ S_h @ u2
print(f"<S u1, u1> = {rayleigh_u1:.4f} (expect 3)")
print(f"<S u2, u2> = {rayleigh_u2:.4f} (expect 1)")

thetas = np.linspace(0, 2*np.pi, 400)
phi = np.array([illumination(np.array([np.cos(t), np.sin(t)]), [h1,h2,h3]) for t in thetas])

theta_u1 = np.arctan2(u1[1], u1[0]) % (2*np.pi)
theta_u2 = np.arctan2(u2[1], u2[0]) % (2*np.pi)

plt.figure(figsize=(8,4))
plt.plot(thetas, phi, label=r'$\varphi(\theta)$')
plt.axvline(theta_u1, color='red', linestyle='--', label=r'$u_1$ direction (max)')
plt.axvline(theta_u2, color='green', linestyle='--', label=r'$u_2$ direction (min)')
plt.xlabel(r'$\theta$')
plt.ylabel(r'$\varphi(\theta)$')
plt.legend()
plt.title('Illumination function with eigenvector directions marked')
plt.show()

## Lab Exercise 3.6 (Optional, Challenge): Pentagonal Frame

(a) Compute $S$, eigenvalues, $A,B$ for the pentagon. (b) Test $m=5,6,7,8$ for tightness. (c) Verify the trace formula and deduce $A=m/2$.

In [ ]:
def regular_polygon(m):
    angles = 2 * np.pi * np.arange(m) / m
    return np.vstack([np.cos(angles), np.sin(angles)])

F_pent = regular_polygon(5)
S_pent = None       # TODO: frame_operator(F_pent)
A_pent, B_pent, eigs_pent, _ = None, None, None, None  # TODO: exact_frame_bounds(F_pent)
print(f"Pentagon: A={A_pent:.4f}, B={B_pent:.4f}, tight={is_tight(F_pent)}")

print("\n=== Testing m=5..8 ===")
for m in [5, 6, 7, 8]:
    F_m = regular_polygon(m)
    tight_m = None  # TODO: is_tight(F_m)
    print(f"m={m}: tight={tight_m}")

print("\n=== Trace formula check ===")
for m in [5, 6, 7, 8]:
    F_m = regular_polygon(m)
    trace_m = None  # TODO: np.trace(frame_operator(F_m))
    print(f"m={m}: trace(S)={trace_m:.4f} (expect {m}), implied A=m/2={m/2:.4f}")

*Your conjecture about which regular polygons give tight frames in R^2:* (replace this text)